## Registro Dinâmico de Cliente no AgentCore Runtime

Este exemplo demonstra como implantar um MCP no AgentCore Runtime que suporta Registro Dinâmico de Cliente.

Para este tutorial, criaremos um exemplo integrado com Auth0 que suporta esse recurso do AgentCore.

Isso pode ser útil para registrar usuários dinamicamente, usando, por exemplo, seu login de mídia social para permitir que os usuários se conectem ao seu MCP.

Neste tutorial, você aprenderá:

* Como criar um servidor MCP com ferramentas
* Como testar seu servidor localmente
* Como configurar seu tenant Auth0 para suportar DCR e adicionar uma API e um aplicativo
* Como implantar seu servidor na AWS, integrado com DCR no Auth0
* Como invocar seu servidor implantado

### Detalhes do Tutorial

| Informação         | Detalhes                                                   |
|:--------------------|:----------------------------------------------------------|
| Tipo de tutorial       | Hospedagem de Ferramentas + DCR no Auth0                             |
| Tipo de ferramenta           | Servidor MCP                                                |
| Componentes do tutorial | Hospedando ferramenta no AgentCore Runtime, Criando um servidor MCP |
| Vertical do tutorial   | Cross-vertical                                            |
| Complexidade do exemplo  | Médio                                                    |
| SDK usado            | SDK Python Amazon BedrockAgentCore e MCP Client        |

### Arquitetura do Tutorial

Neste tutorial, descreveremos como implantar este exemplo no AgentCore Runtime.

Para fins de demonstração, usaremos um servidor MCP muito simples com 3 ferramentas: `add_numbers`, `multiply_numbers` e `greet_users`.

![Architecture](images/architecture.png)

### Principais Recursos do Tutorial

* Hospedagem de Servidor MCP
* Registro Dinâmico de Cliente (DCR)
* Auth0

In [ ]:
!pip install -Uq -r requirements.txt

**Reinicie seu kernel para refletir as dependências instaladas.**

## Configurando o Auth0

### Configurações Gerais

Neste exemplo, trabalharemos com Auth0 para implementar DCR (Registro Dinâmico de Cliente). Antes de começar, vamos habilitar a opção DCR no console do Auth0.

- No menu do painel esquerdo, clique em `Settings`. Em seguida, em **Tenant Settings**, clique na opção `Advanced`. Role para baixo até encontrar a opção *Dynamic Client Registration (DCR)* e clique nela para *habilitar*. Também clique em *Enable Application Connections*, para que as conexões sejam habilitadas quando um novo aplicativo for criado:

![Enable DCR](images/01_enable_dcr.png)

Para mais informações, consulte a documentação do Auth0 sobre [registro dinâmico de cliente](https://auth0.com/docs/get-started/applications/dynamic-client-registration).

Agora que temos habilitado, vamos usar *Google / Gmail* Identity como o mecanismo de login para usuários (aplicativos) em nosso MCP.

- No menu do painel esquerdo, clique em `Authentication` e depois clique em `Social`. Na tela **Social Connections**, clique na opção `google-oauth2`. Na tela de configurações, role para baixo até encontrar a opção `Promote Connection to Domain Level` e habilite-a:

![Enable Social](images/02_enable_social.png)

Agora você configurou todas as configurações para seu tenant Auth0. Vamos continuar e começar criando uma API Auth0.

---

### Criando uma API

Vamos criar uma API. No menu do painel esquerdo, na seção Applications, clique em `APIs`. Na tela APIs, clique no botão superior direito `+ Create API`.

![API](images/03_create_api.png)

- Name: O nome da sua API.
- Identifier: Identificador único para a API. Este valor será usado como o parâmetro `audience` nas chamadas de autorização.
- Você pode manter as outras opções como padrão e clicar em Create.

Nas configurações da sua API, clique na aba `Permissions` e adicione a permissão invoke seguindo este padrão `<identifier_name>:Invoke`:

![Permissions](images/04_permissions.png)

Pronto. Agora sua API está configurada. Vamos criar um aplicativo.

---

### Criando um Aplicativo

Vamos finalizar nossa configuração criando um aplicativo. No menu do painel esquerdo `Applications`, clique em `Applications` e depois em `+ Create Application`.

Na tela de criação do aplicativo, dê um nome ao aplicativo, escolha a opção `Machine-to-Machine` e clique em criar:

![App](images/05_app.png)

Em seguida, na próxima tela, você será solicitado a selecionar uma API. Selecione a API que você criou na etapa anterior, clique no botão `all` para autorizar todas as ações e clique no botão `Authorize`:

![Auth](images/06_authorizing.png)

Agora seu aplicativo está criado.

---

### Obtendo Informações do Aplicativo

Agora precisamos obter as informações do aplicativo para que possamos usá-las em nosso exemplo.

Clique no seu aplicativo e vá para a página `quickstart`. Esta página mostrará um exemplo Linux `curl` com todos os parâmetros necessários. Copie o domínio do seu valor `--url`, como o seguinte exemplo:

Seu parâmetro `--url`:

```bash
--url https://<your-auth0-tenant>.us.auth0.com/oauth/token
```

Construa a seguinte variável de ambiente que será usada neste notebook:

```bash
export DISCOVERY_URL="https://<your-auth0-tenant>.us.auth0.com/.well-known/openid-configuration"
```

**Substitua aqui com seu domínio**

In [ ]:
DISCOVERY_URL="https://<your-auth0-tenant>.us.auth0.com/.well-known/openid-configuration"

Também precisamos preencher nosso audience. Será o identificador da nossa API. Se você seguir este exemplo e usar `ac-runtime-api`, esse será o valor de audience:

In [ ]:
# Substitua se você criou com um identificador diferente
AUDIENCE="ac-runtime-api"

### Criando Servidor MCP

Agora vamos criar nosso servidor MCP com três ferramentas simples:

In [ ]:
%%writefile server.py
from mcp.server.fastmcp import FastMCP
from starlette.responses import JSONResponse

mcp = FastMCP(host="0.0.0.0", stateless_http=True)

@mcp.tool()
def add_numbers(a: int, b: int) -> int:
    """Add two numbers together"""
    return a + b

@mcp.tool()
def multiply_numbers(a: int, b: int) -> int:
    """Multiply two numbers together"""
    return a * b

@mcp.tool()
def greet_user(name: str) -> str:
    """Greet a user by name"""
    return f"Hello, {name}! Nice to meet you."

if __name__ == "__main__":
    mcp.run(transport="streamable-http")

### Teste Local do Servidor MCP (opcional)

Se você quiser, pode testar seu novo servidor MCP localmente.

Execute a célula a seguir para gerar o script Python de teste.

In [ ]:
%%writefile test_my_mcp_client.py
import asyncio
from datetime import timedelta

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

async def main():
    mcp_url = "http://localhost:8000/mcp"
    headers = {}

    async with streamablehttp_client(mcp_url, headers, timeout=timedelta(seconds=120), terminate_on_close=False) as (
        read_stream,
        write_stream,
        _,
    ):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            tool_result = await session.list_tools()
            print("Available tools:")
            for tool in tool_result.tools:
                print(f"  - {tool.name}: {tool.description}")

if __name__ == "__main__":
    asyncio.run(main())

Para testar seu servidor MCP localmente:

1. **Terminal 1**: Inicie o servidor MCP

```bash
python server.py
```

2. **Terminal 2**: Execute o script de teste

```bash
python test_my_mcp_client.py
```

### Lançando Servidor MCP no AgentCore Runtime

Agora vamos criar um aplicativo AgentCore Runtime usando a URL e audience fornecidos.

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
boto_session = Session()
region = boto_session.region_name

agentcore_runtime = Runtime()
agent_name = "mcp_dcr_sample"

auth_config = {
    "customJWTAuthorizer": {
        "allowedAudience": [
            AUDIENCE
        ],
        "discoveryUrl": DISCOVERY_URL,
    }
}

response = agentcore_runtime.configure(
    entrypoint="server.py",
    auto_create_execution_role=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name,
    authorizer_configuration=auth_config,
    protocol="MCP",
    memory_mode="NO_MEMORY",
    deployment_type="direct_code_deploy",
    runtime_type="PYTHON_3_13",
)

In [ ]:
print("Lançando servidor MCP no AgentCore Runtime...")
print("Isso pode levar vários minutos...")
launch_result = agentcore_runtime.launch()
print("Lançamento concluído ✓")
print(f"Agent ARN: {launch_result.agent_arn}")
print(f"Agent ID: {launch_result.agent_id}")

### Testando

O arquivo `mcp_auth0_client.py` contém uma implementação local para conectar ao nosso agente implantado no AgentCore Runtime.

Quando você executar seu script, ele o redirecionará para uma página de login que solicitará sua identidade do Google para fazer login.

Depois de fazer login, ele pedirá que você autorize o Auth0 a criar um aplicativo usando seu e-mail.

![Redirect](images/07_redirect.png)

Se você aceitar, ele redirecionará para uma página HTML simples (implementada em `mcp_auth0_client.py`) mostrando que foi bem-sucedido.

![OK](images/08_success.png)

Definindo variáveis de ambiente que o script está esperando

In [ ]:
agent_arn = launch_result.agent_arn
region = "us-west-2"
custom_endpoint =f"https://bedrock-agentcore.{region}.amazonaws.com"

agent_arn, custom_endpoint, AUDIENCE

In [ ]:
import mcp_auth0_client as mcp_client

await mcp_client.main(agent_arn, custom_endpoint, AUDIENCE)

## Limpeza

In [ ]:
agentcore_runtime.destroy()

### 🎉 Parabéns!

Você concluiu com sucesso:

- **Criou um tenant** no Auth0
- **Habilitou DCR** no Auth0 e criou uma API e um aplicativo
- Criou um **servidor MCP** com ferramentas personalizadas
- **Testou localmente** usando um cliente MCP
- Configurou **autenticação com DCR**
- Implantou na AWS usando **AgentCore Runtime**
- **Invocou remotamente** com autenticação adequada
- **Aprendeu** conceitos e melhores práticas de MCP

Seu servidor MCP agora está em execução no Amazon Bedrock AgentCore Runtime e pronto para uso em produção!